In [0]:

# Objective: Ingest raw KKBox CSVs to Bronze Delta tables


LANDING= "/Volumes/churn_project/bronze/landing"

BRONZE_DB= "churn_project.bronze" #catalog.schema

FILES = {
    # Training cohort (Feb expiry users, labels from train_v2)
    "train_v2":             f"{LANDING}/train_v2.csv",
    "user_logs_train":      f"{LANDING}/user_logs_trimmed.csv",
    "transactions_train":   f"{LANDING}/transactions_trimmed.csv",

    # Scoring cohort (March-active users, no labels)
    "user_logs_score":      f"{LANDING}/user_logs_v2.csv",
    "transactions_score":   f"{LANDING}/transactions_v2.csv",

    # Demographics — applies to both cohorts
    "members":              f"{LANDING}/members_v3.csv",
}

print("Landing volume contents:")
for name, path in FILES.items():
    info = dbutils.fs.ls(path)
    size_mb = info[0].size / (1024**2)
    print(f"  {name:<25} {size_mb:>8.1f} MB  →  {path.split('/')[-1]}")

Landing volume contents:
  train_v2                      43.5 MB  →  train_v2.csv
  user_logs_train             1818.5 MB  →  user_logs_trimmed.csv
  transactions_train           135.5 MB  →  transactions_trimmed.csv
  user_logs_score             1365.2 MB  →  user_logs_v2.csv
  transactions_score           110.0 MB  →  transactions_v2.csv
  members                      408.1 MB  →  members_v3.csv


#### train
the train set, containing the user ids and whether they have churned.

- msno: user id
- is_churn: This is the target variable. Churn is defined as whether the user did not continue the subscription within 30 days of expiration. is_churn = 1 means churn,is_churn = 0 means renewal.

#### transactions

- msno: user id
- payment_method_id: payment method
- payment_plan_days: length of membership plan in days
- plan_list_price: in New Taiwan Dollar (NTD)
- actual_amount_paid: in New Taiwan Dollar (NTD)
- is_auto_renew
- transaction_date: format %Y%m%d
- membership_expire_date: format %Y%m%d
- is_cancel: whether or not the user canceled the membership in this transaction.


#### users_logs

- msno: user id
- date: format %Y%m%d
- num_25: # of songs played less than 25% of the song length
- num_50: # of songs played between 25% to 50% of the song length
- num_75: # of songs played between 50% to 75% of of the song length
- num_985: # of songs played between 75% to 98.5% of the song length
- num_100: # of songs played over 98.5% of the song length
- num_unq: # of unique songs played
- total_secs: total seconds played


#### members

- msno
- city
- bd: age. 
- gender
- registered_via: registration method
- registration_init_time: format %Y%m%d
- expiration_date: format %Y%m%d, taken as a snapshot at which the member.csv is extracted. Not representing the actual churn behavior.




In [0]:
df_train_raw = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(FILES["train_v2"])
)

df_train_raw.printSchema()
print(f"\nRow count: {df_train_raw.count():,}")

print("\nSample rows:")
df_train_raw.show(5, truncate=False)

print("Value counts for is_churn:")
df_train_raw.groupBy("is_churn").count().orderBy("is_churn").show()

root
 |-- msno: string (nullable = true)
 |-- is_churn: integer (nullable = true)


Row count: 970,960

Sample rows:
+--------------------------------------------+--------+
|msno                                        |is_churn|
+--------------------------------------------+--------+
|ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=|1       |
|f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=|1       |
|zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=|1       |
|8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=|1       |
|K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=|1       |
+--------------------------------------------+--------+
only showing top 5 rows
Value counts for is_churn:
+--------+------+
|is_churn| count|
+--------+------+
|       0|883630|
|       1| 87330|
+--------+------+



In [0]:
df_members_raw = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(FILES["members"])
)

df_members_raw.printSchema()
print(f"\nRow count: {df_members_raw.count():,}")

print("\nSample rows:")
df_members_raw.show(5, truncate=False)

root
 |-- msno: string (nullable = true)
 |-- city: integer (nullable = true)
 |-- bd: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- registered_via: integer (nullable = true)
 |-- registration_init_time: integer (nullable = true)


Row count: 6,769,473

Sample rows:
+--------------------------------------------+----+---+------+--------------+----------------------+
|msno                                        |city|bd |gender|registered_via|registration_init_time|
+--------------------------------------------+----+---+------+--------------+----------------------+
|Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=|1   |0  |NULL  |11            |20110911              |
|+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=|1   |0  |NULL  |7             |20110914              |
|cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=|1   |0  |NULL  |11            |20110915              |
|9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=|1   |0  |NULL  |11            |20110915              |
|WF

In [0]:
print("\nbd (age) distribution:")
df_members_raw.selectExpr(
    "min(bd)                            as min_age",
    "max(bd)                            as max_age",
    "percentile(bd, 0.25)               as p25",
    "percentile(bd, 0.50)               as p50",
    "percentile(bd, 0.75)               as p75",
    "sum(case when bd <= 0 then 1 end)  as nonsense_low",
    "sum(case when bd > 100 then 1 end) as nonsense_high",
    "count(*)                           as total_rows"
).show()

print("\ngender distribution:")
df_members_raw.groupBy("gender").count().orderBy("count", ascending=False).show()

print("\nregistered_via distribution:")
df_members_raw.groupBy("registered_via").count().orderBy("count", ascending=False).show()



bd (age) distribution:
+-------+-------+---+---+----+------------+-------------+----------+
|min_age|max_age|p25|p50| p75|nonsense_low|nonsense_high|total_rows|
+-------+-------+---+---+----+------------+-------------+----------+
|  -7168|   2016|0.0|0.0|21.0|     4540489|         5377|   6769473|
+-------+-------+---+---+----+------------+-------------+----------+


gender distribution:
+------+-------+
|gender|  count|
+------+-------+
|  NULL|4429505|
|  male|1195355|
|female|1144613|
+------+-------+


registered_via distribution:
+--------------+-------+
|registered_via|  count|
+--------------+-------+
|             4|2793213|
|             3|1643208|
|             9|1482863|
|             7| 805895|
|            11|  25047|
|            13|   5455|
|             8|   3982|
|             5|   3115|
|            17|   1494|
|             2|   1452|
|             6|   1213|
|            19|    974|
|            16|    888|
|            14|    615|
|             1|     43|
|       

There are negative and very large age values which will need to be dealt with.

In [0]:
# transactions_trimmed is 135 MB — inferSchema acceptable.
df_txn_train_raw = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(FILES["transactions_train"])
)

df_txn_train_raw.printSchema()

print(f"\nRow count: {df_txn_train_raw.count():,}")

print("\nSample rows:")
df_txn_train_raw.show(5, truncate=False)

root
 |-- msno: string (nullable = true)
 |-- payment_method_id: integer (nullable = true)
 |-- payment_plan_days: integer (nullable = true)
 |-- plan_list_price: integer (nullable = true)
 |-- actual_amount_paid: integer (nullable = true)
 |-- is_auto_renew: integer (nullable = true)
 |-- transaction_date: integer (nullable = true)
 |-- membership_expire_date: integer (nullable = true)
 |-- is_cancel: integer (nullable = true)


Row count: 1,771,849

Sample rows:
+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+
|msno                                        |payment_method_id|payment_plan_days|plan_list_price|actual_amount_paid|is_auto_renew|transaction_date|membership_expire_date|is_cancel|
+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+-------------------

In [0]:
print("\nDate column ranges:")
df_txn_train_raw.selectExpr(
    "min(transaction_date) as earliest_txn",
    "max(transaction_date) as latest_txn",
    "min(membership_expire_date) as earliest_exp",
    "max(membership_expire_date) as latest_exp"
).show()

print("\n Amount paid distribution")
df_txn_train_raw.selectExpr(
    "min(actual_amount_paid)                         as min_paid",
    "max(actual_amount_paid)                         as max_paid",
    "sum(case when actual_amount_paid = 0 then 1 end) as zero_paid_count",
    "count(*)                                         as total_rows"
).show()

print("\nis_cancel and is_auto_renew breakdown:")
df_txn_train_raw.groupBy("is_cancel", "is_auto_renew").count().orderBy("is_cancel", "is_auto_renew").show()



Date column ranges:
+------------+----------+------------+----------+
|earliest_txn|latest_txn|earliest_exp|latest_exp|
+------------+----------+------------+----------+
|    20170101|  20170228|    19700101|  20170331|
+------------+----------+------------+----------+


 Amount paid distribution
+--------+--------+---------------+----------+
|min_paid|max_paid|zero_paid_count|total_rows|
+--------+--------+---------------+----------+
|       0|     400|           6124|   1771849|
+--------+--------+---------------+----------+


is_cancel and is_auto_renew breakdown:
+---------+-------------+-------+
|is_cancel|is_auto_renew|  count|
+---------+-------------+-------+
|        0|            0| 148334|
|        0|            1|1593793|
|        1|            1|  29722|
+---------+-------------+-------+



In [0]:
# Peek at user_logs. We won't use Inferschema as it forces a full scan to determine types. Instead
# we will read as all-string with header to see column names.

df_logs_train_raw = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", False)   # strings only — no full scan
    .load(FILES["user_logs_train"])
)


df_logs_train_raw.printSchema()

print("\nSample rows:")
df_logs_train_raw.show(5, truncate=False)

print(f"\nRow count: {df_logs_train_raw.count():,}")
print(f"Distinct users: {df_logs_train_raw.select('msno').distinct().count():,}")

root
 |-- msno: string (nullable = true)
 |-- date: string (nullable = true)
 |-- num_25: string (nullable = true)
 |-- num_50: string (nullable = true)
 |-- num_75: string (nullable = true)
 |-- num_985: string (nullable = true)
 |-- num_100: string (nullable = true)
 |-- num_unq: string (nullable = true)
 |-- total_secs: string (nullable = true)


Sample rows:
+--------------------------------------------+--------+------+------+------+-------+-------+-------+----------+
|msno                                        |date    |num_25|num_50|num_75|num_985|num_100|num_unq|total_secs|
+--------------------------------------------+--------+------+------+------+-------+-------+-------+----------+
|MZhYC2aGSvFk0hmJQa+fBlADmopI2ONFeXTGGjCzh40=|20170222|2     |0     |1     |0      |90     |30     |20488.607 |
|zcOrKXPthN+uU4tDC0h/Cf1m2TGGzE+rv08hP5LqgXg=|20170125|2     |0     |1     |0      |35     |36     |8066.38   |
|FwN8s/3raOheGkz7mEnUi/VBr/lVUDF/Kxp/AyBTnqo=|20170105|4     |2     |1     

In [0]:
print("\nDate range:")
df_logs_train_raw.selectExpr(
    "min(date) as earliest_log",
    "max(date) as latest_log"
).show()

print("\nSample total_secs values:")
df_logs_train_raw.selectExpr(
    "min(cast(total_secs as double)) as min_secs",
    "max(cast(total_secs as double)) as max_secs",
    "sum(case when total_secs is null then 1 end) as null_secs"
).show()


Date range:
+------------+----------+
|earliest_log|latest_log|
+------------+----------+
|    20170101|  20170228|
+------------+----------+


Sample total_secs values:
+--------+-----------+---------+
|min_secs|   max_secs|null_secs|
+--------+-----------+---------+
|   0.001|2833307.509|     NULL|
+--------+-----------+---------+



### Findings from initial exploration
- All date columns were inferred as integers instead of proper date types:
- 6,124 zero-payment transactions (0.35% of all transactions)
- Price range: 0–400 NTD
- These $0 transactions likely represent free trials, promotional offers, or error records worth investigating?
- Missing combination: is_cancel=1, is_auto_renew=0 doesn't exist. This suggests users can only cancel if auto-renew was enabled.
- The earliest expiration date is 19700101 (Unix epoch) — likely a placeholder for null/missing dates rather than real 1970 memberships.

In [0]:
df_zero_paid= df_txn_train_raw.filter(df_txn_train_raw.actual_amount_paid == 0)
print(f"Zero-payment transaction count: {df_zero_paid.count():,}")

print("\nis_cancel / is_auto_renew breakdown for zero-payment transactions:")
df_zero_paid.groupBy("is_cancel", "is_auto_renew").count().show()

print("\npayment_plan_days distribution for zero-payment transactions:")
df_zero_paid.groupBy("payment_plan_days").count().orderBy("count", ascending=False).show(10)

print("\nplan_list_price vs actual_amount_paid — are these promos (list price > 0) or genuinely free plans (list price = 0 too)?")
df_zero_paid.groupBy("plan_list_price").count().orderBy("count", ascending=False).show(10)



Zero-payment transaction count: 6,124

is_cancel / is_auto_renew breakdown for zero-payment transactions:
+---------+-------------+-----+
|is_cancel|is_auto_renew|count|
+---------+-------------+-----+
|        0|            0| 1307|
|        0|            1| 4793|
|        1|            1|   24|
+---------+-------------+-----+


payment_plan_days distribution for zero-payment transactions:
+-----------------+-----+
|payment_plan_days|count|
+-----------------+-----+
|               30| 4824|
|                7|  675|
|               60|  289|
|                1|  144|
|              240|   83|
|              120|   49|
|               10|   29|
|              195|   11|
|              395|    9|
|              410|    6|
+-----------------+-----+
only showing top 10 rows

plan_list_price vs actual_amount_paid — are these promos (list price > 0) or genuinely free plans (list price = 0 too)?
+---------------+-----+
|plan_list_price|count|
+---------------+-----+
|            149| 3299|


In [0]:
print("How many have placeholder membership_expire_date= 19700101?")
df_placeholder_date= df_txn_train_raw.filter(df_txn_train_raw.membership_expire_date == 19700101)
print(f"Placeholder date count: {df_placeholder_date.count():,}")

print("\nFor these rows, what does transaction_date look like? Recent or also placeholder?")
df_placeholder_date.selectExpr(
    "min(transaction_date) as earliest_txn",
    "max(transaction_date) as latest_txn"
).show()

print("\nis_cancel breakdown for these rows")
df_placeholder_date.groupBy("is_cancel").count().show()


How many have placeholder membership_expire_date= 19700101?
Placeholder date count: 200

For these rows, what does transaction_date look like? Recent or also placeholder?
+------------+----------+
|earliest_txn|latest_txn|
+------------+----------+
|    20170110|  20170220|
+------------+----------+


is_cancel breakdown for these rows
+---------+-----+
|is_cancel|count|
+---------+-----+
|        0|  198|
|        1|    2|
+---------+-----+



- Zero-payment transactions: mostly promos (plan_list_price of 149/180/120 NTD but actual_amount_paid = 0 — full discount applied), plus 1,307 genuinely free plans (plan_list_price = 0 too). We keep all of these rows as-is; they're real transactions, just discounted or free.

- 1970 epoch dates: 200 rows out of 1.77M (0.01%), negligible volume. These are real, recent transactions (Jan to Feb 2017) where membership_expire_date failed to populate, defaulting to the Unix epoch placeholder rather than representing an actual 1970 membership. Since is_cancel is false for 198 of them, these aren't cancellation-driven nulls either. Likely a data entry/system gap. I'll null these out when parsing dates.

In [0]:
# transactions_v2 — scoring cohort, March activity.
# Same checks as transactions_train, since this feeds the same feature pipeline.

df_txn_score_raw = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(FILES["transactions_score"])
)

df_txn_score_raw.printSchema()

print(f"\nRow count: {df_txn_score_raw.count():,}")
print(f"Distinct users: {df_txn_score_raw.select('msno').distinct().count():,}")

print("\nDate ranges:")
df_txn_score_raw.selectExpr(
    "min(transaction_date)       as txn_date_min",
    "max(transaction_date)       as txn_date_max",
    "min(membership_expire_date) as exp_date_min",
    "max(membership_expire_date) as exp_date_max"
).show()

print("\nZero-payment check:")
df_txn_score_raw.selectExpr(
    "sum(case when actual_amount_paid = 0 then 1 end) as zero_paid_count",
    "count(*) as total_rows"
).show()

print("\nEpoch placeholder check:")
df_txn_score_raw.filter(col("membership_expire_date") == 19700101).count()

root
 |-- msno: string (nullable = true)
 |-- payment_method_id: integer (nullable = true)
 |-- payment_plan_days: integer (nullable = true)
 |-- plan_list_price: integer (nullable = true)
 |-- actual_amount_paid: integer (nullable = true)
 |-- is_auto_renew: integer (nullable = true)
 |-- transaction_date: integer (nullable = true)
 |-- membership_expire_date: integer (nullable = true)
 |-- is_cancel: integer (nullable = true)


Row count: 1,431,009
Distinct users: 1,197,050

Date ranges:
+------------+------------+------------+------------+
|txn_date_min|txn_date_max|exp_date_min|exp_date_max|
+------------+------------+------------+------------+
|    20150101|    20170331|    20160419|    20361015|
+------------+------------+------------+------------+


Zero-payment check:
+---------------+----------+
|zero_paid_count|total_rows|
+---------------+----------+
|          21448|   1431009|
+---------------+----------+


Epoch placeholder check:


0

In [0]:
# user_logs_v2 — scoring cohort, 1.3 GB. Same careful approach as training logs.

df_logs_score_raw = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", False) #do not want to read full data
    .load(FILES["user_logs_score"])
)


df_logs_score_raw.printSchema()

print(f"\nRow count: {df_logs_score_raw.count():,}")
print(f"Distinct users: {df_logs_score_raw.select('msno').distinct().count():,}")

print("\nDate range:")
df_logs_score_raw.selectExpr(
    "min(date) as earliest_log",
    "max(date) as latest_log"
).show()

root
 |-- msno: string (nullable = true)
 |-- date: string (nullable = true)
 |-- num_25: string (nullable = true)
 |-- num_50: string (nullable = true)
 |-- num_75: string (nullable = true)
 |-- num_985: string (nullable = true)
 |-- num_100: string (nullable = true)
 |-- num_unq: string (nullable = true)
 |-- total_secs: string (nullable = true)


Row count: 18,396,362
Distinct users: 1,103,894

Date range:
+------------+----------+
|earliest_log|latest_log|
+------------+----------+
|    20170301|  20170331|
+------------+----------+



In [0]:
print("total_secs distribution (training logs):")
df_logs_train_raw.selectExpr(
    "min(cast(total_secs as double))    as min_secs",
    "max(cast(total_secs as double))    as max_secs",
    "avg(cast(total_secs as double))    as avg_secs"
).show()

print("\nAny negative or zero total_secs rows? (would indicate skipped/instant plays)")
df_logs_train_raw.filter(col("total_secs").cast("double") <= 0).count()

print("\nnum_unq distribution:")
df_logs_train_raw.selectExpr(
    "min(cast(num_unq as int)) as min_unq",
    "max(cast(num_unq as int)) as max_unq"
).show()

total_secs distribution (training logs):
+--------+-----------+-----------------+
|min_secs|   max_secs|         avg_secs|
+--------+-----------+-----------------+
|   0.001|2833307.509|7895.982873242157|
+--------+-----------+-----------------+


Any negative or zero total_secs rows? (would indicate skipped/instant plays)

num_unq distribution:
+-------+-------+
|min_unq|max_unq|
+-------+-------+
|      1|   3869|
+-------+-------+



- txn_date_min = 20150101 even though we only uploaded transactions_v2.csv (which should be March activity). This isn't a problem. transactions_v2.csv is described as "transactions up to 3/31/2017" without a lower bound, so it includes full history for users who happened to transact in March. We're not filtering this file by date the way we did the training file, so this is expected. We'll need to decide in future step what window of transactions_score history to actually use for feature engineering.

- exp_date_max = 20361015. A membership expiring in 2036 is unusual but not impossible.

- Zero-payment rate jumped: Training had 6,124 zero-paid out of 1.77M (0.35%). Scoring has 21,448 out of 1.43M- that's 1.5%, over 4x higher. Worth understanding why.

In [0]:
# Check how many transactions have unusually far-future expiration dates
print("Transactions with membership_expire_date beyond 2020:")
df_txn_score_raw.filter(col("membership_expire_date") > 20201231).count()

print("\nSample of these far-future rows:")
df_txn_score_raw.filter(col("membership_expire_date") > 20201231).show(10, truncate=False)

print("\npayment_plan_days for these- are they legitimately long plans?")
df_txn_score_raw.filter(col("membership_expire_date") > 20201231).groupBy("payment_plan_days").count().orderBy("count", ascending=False).show(10)

Transactions with membership_expire_date beyond 2020:

Sample of these far-future rows:
+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+
|msno                                        |payment_method_id|payment_plan_days|plan_list_price|actual_amount_paid|is_auto_renew|transaction_date|membership_expire_date|is_cancel|
+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+
|GUlOynwcbchLoK1DjZ9pMqC3HiK1BOGDV3g4xuUeNA0=|41               |30               |100            |100               |1            |20150721        |20210629              |0        |
|IfrVgVmLLOSZbiFNN42GIiOzrz4onvGPNUbCK96vEVA=|38               |30               |149            |149               |0            |20170208        |20240311              |0        |
|L

In [0]:
print("\nplan_list_price breakdown for scoring cohort's zero-payment rows:")
df_txn_score_raw.filter(col("actual_amount_paid") == 0).groupBy("plan_list_price").count().orderBy("count", ascending=False).show(10)

print("\nDoes the zero-payment rate vary by transaction month? (training was Jan-Feb only, scoring spans 2015-2017)")
df_txn_score_raw.filter(col("actual_amount_paid") == 0).selectExpr(
    "cast(transaction_date / 10000 as int) as year"
).groupBy("year").count().orderBy("year").show()


plan_list_price breakdown for scoring cohort's zero-payment rows:
+---------------+-----+
|plan_list_price|count|
+---------------+-----+
|              0|16197|
|            180| 3148|
|            149| 2023|
|            120|   50|
|             99|   15|
|           1599|    8|
|            129|    6|
|            894|    1|
+---------------+-----+


Does the zero-payment rate vary by transaction month? (training was Jan-Feb only, scoring spans 2015-2017)
+----+-----+
|year|count|
+----+-----+
|2015|  156|
|2016| 3379|
|2017|17913|
+----+-----+



- The 30-day plan dominates (801 of the far-future-expiry rows) — these are likely standard monthly subscribers whose membership_expire_date got corrupted or set incorrectly, not legitimate multi-year plans. A 30-day plan with an expiry in 2036 doesn't make sense. It's a data quality issue. I'll cap or flag these.

- Zero-payment rate is concentrated in 2017 (17,913 of 21,448 — 83.5%) and dominated by plan_list_price = 0 (16,197 of 21,448 — 75.5%). This is a different pattern than training. In training, zero-payment was mostly discounted paid plans (price 149/180 reduced to 0). In scoring, it's mostly genuinely free plans (price already 0). I will keep all rows as real transactions (same call as training). plan_list_price = 0 users are free-tier; actual_amount_paid = 0 on a priced plan is a promo.

In [0]:
zero_secs_count = df_logs_train_raw.filter(col("total_secs").cast("double") <= 0).count()
print(f"Rows with total_secs <= 0: {zero_secs_count:,}")

zero_secs_score = df_logs_score_raw.filter(col("total_secs").cast("double") <= 0).count()
print(f"Rows with total_secs <= 0 (scoring): {zero_secs_score:,}")

Rows with total_secs <= 0: 0
Rows with total_secs <= 0 (scoring): 0


## Exploration Summary — Findings Before Finalizing Schema

Before defining explicit schemas, we explored all six raw files to understand
their structure, quality, and quirks. Key findings and how each is handled:

| Finding | File(s) | Handling |
|---|---|---|
| Dates are `yyyyMMdd` integers, not native dates | All except `train_v2` | Cast to `DateType` during date-parsing step |
| `bd` (age) contains dirty values (negatives, >100) | `members` | Left as `IntegerType` here; cleaned/nulled in Step 3 feature engineering |
| Zero-payment transactions exist — mix of promos (discounted paid plans) and free-tier plans | `transactions_train` (0.35%), `transactions_score` (1.5%) | Kept as real transactions in both cohorts. Rate differs ~4x between cohorts — likely reflects a larger free-trial promotion concentrated in early 2017, which falls inside the scoring window. Noted, not treated as an error. |
| `is_cancel = 1` only co-occurs with `is_auto_renew = 1` in observed data | `transactions_train` | Documented as an observed pattern (cancellation appears tied to auto-renew being active), not enforced as a hard rule downstream |
| 200 rows with placeholder expiry date `19700101` (Unix epoch) | `transactions_train` | Treated as missing/null, not a real date — nulled out during date parsing to avoid corrupting `days_until_expiration` |
| Far-future expiry dates (some extending to 2036), concentrated in 30-day plans | `transactions_score` | Flagged as a likely data quality artifact (a 30-day plan shouldn't expire in 2036). Capped/handled explicitly in Step 3 feature engineering, not silently passed through |
| `total_secs` has zero rows with zero or negative values | `user_logs_train`, `user_logs_score` | No cleaning needed — confirms the column is well-formed |
| `transactions_score` spans 2015–2017, not just March | `transactions_score` | Expected, since this file is not date-filtered like the training file. Step 3 will decide what lookback window to use for scoring features |


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType
)

# train_v2: confirmed clean — msno + is_churn
schema_train = StructType([
    StructField("msno",     StringType(),  nullable=False),
    StructField("is_churn", IntegerType(), nullable=False),
])

# transactions: same structure for both train and score cohorts
# Dates kept as IntegerType here — parsed to DateType later
schema_transactions = StructType([
    StructField("msno",                    StringType(),  nullable=False),
    StructField("payment_method_id",       IntegerType(), nullable=True),
    StructField("payment_plan_days",       IntegerType(), nullable=True),
    StructField("plan_list_price",         IntegerType(), nullable=True),
    StructField("actual_amount_paid",      IntegerType(), nullable=True),
    StructField("is_auto_renew",           IntegerType(), nullable=True),
    StructField("transaction_date",        IntegerType(), nullable=True),
    StructField("membership_expire_date",  IntegerType(), nullable=True),
    StructField("is_cancel",               IntegerType(), nullable=True),
])

# user_logs: same structure for both train and score cohorts
# total_secs confirmed clean (no zero/negative)
schema_user_logs = StructType([
    StructField("msno",       StringType(),  nullable=False),
    StructField("date",       IntegerType(), nullable=True),
    StructField("num_25",     IntegerType(), nullable=True),
    StructField("num_50",     IntegerType(), nullable=True),
    StructField("num_75",     IntegerType(), nullable=True),
    StructField("num_985",    IntegerType(), nullable=True),
    StructField("num_100",    IntegerType(), nullable=True),
    StructField("num_unq",    IntegerType(), nullable=True),
    StructField("total_secs", DoubleType(),  nullable=True),
])

# members: bd kept as IntegerType, cleaned downstream in Step 3
schema_members = StructType([
    StructField("msno",                    StringType(),  nullable=False),
    StructField("city",                    IntegerType(), nullable=True),
    StructField("bd",                      IntegerType(), nullable=True),
    StructField("gender",                  StringType(),  nullable=True),
    StructField("registered_via",          IntegerType(), nullable=True),
    StructField("registration_init_time",  IntegerType(), nullable=True),
])



In [0]:
df_train = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema_train)
    .load(FILES["train_v2"])
)

df_members = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema_members)
    .load(FILES["members"])
)

df_txn_train = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema_transactions)
    .load(FILES["transactions_train"])
)

df_txn_score = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema_transactions)
    .load(FILES["transactions_score"])
)

df_logs_train = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema_user_logs)
    .load(FILES["user_logs_train"])
)

df_logs_score = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema_user_logs)
    .load(FILES["user_logs_score"])
)

print("All six DataFrames loaded with explicit schemas.")
print(f"  train             : {df_train.count():>10,} rows")
print(f"  members           : {df_members.count():>10,} rows")
print(f"  transactions_train: {df_txn_train.count():>10,} rows")
print(f"  transactions_score: {df_txn_score.count():>10,} rows")
print(f"  user_logs_train   : {df_logs_train.count():>10,} rows")
print(f"  user_logs_score   : {df_logs_score.count():>10,} rows")

All six DataFrames loaded with explicit schemas.
  train             :    970,960 rows
  members           :  6,769,473 rows
  transactions_train:  1,771,849 rows
  transactions_score:  1,431,009 rows
  user_logs_train   : 24,884,219 rows
  user_logs_score   : 18,396,362 rows


In [0]:
from pyspark.sql.functions import to_date, col, when

def parse_kkbox_date(df, col_name, treat_as_null_if=None):
    """
    Cast a yyyyMMdd integer column to DateType.
    Null out specific placeholder values before parsing
    (e.g. the 19700101 epoch placeholder we found in transactions_train).
    """
    if treat_as_null_if is not None:
        df = df.withColumn(
            col_name,
            when(col(col_name) == treat_as_null_if, None).otherwise(col(col_name))
        )
    return df.withColumn(
        col_name,
        to_date(col(col_name).cast("string"), "yyyyMMdd")
    )

df_txn_train = parse_kkbox_date(df_txn_train, "transaction_date")
df_txn_train = parse_kkbox_date(df_txn_train, "membership_expire_date", treat_as_null_if=19700101)

df_txn_score = parse_kkbox_date(df_txn_score, "transaction_date")
df_txn_score = parse_kkbox_date(df_txn_score, "membership_expire_date", treat_as_null_if=19700101)

df_logs_train = parse_kkbox_date(df_logs_train, "date")
df_logs_score = parse_kkbox_date(df_logs_score, "date")

df_members = parse_kkbox_date(df_members, "registration_init_time")



In [0]:
df_txn_train.selectExpr(
    "min(transaction_date)       as txn_date_min",
    "max(transaction_date)       as txn_date_max",
    "min(membership_expire_date) as exp_date_min",
    "max(membership_expire_date) as exp_date_max",
    "sum(case when membership_expire_date is null then 1 else 0 end) as null_expiry_count"
).show()

+------------+------------+------------+------------+-----------------+
|txn_date_min|txn_date_max|exp_date_min|exp_date_max|null_expiry_count|
+------------+------------+------------+------------+-----------------+
|  2017-01-01|  2017-02-28|  2016-12-29|  2017-03-31|              200|
+------------+------------+------------+------------+-----------------+



In [0]:
df_txn_score.selectExpr(
    "min(transaction_date)       as txn_date_min",
    "max(transaction_date)       as txn_date_max",
    "min(membership_expire_date) as exp_date_min",
    "max(membership_expire_date) as exp_date_max",
    "sum(case when membership_expire_date is null then 1 else 0 end) as null_expiry_count"
).show()

+------------+------------+------------+------------+-----------------+
|txn_date_min|txn_date_max|exp_date_min|exp_date_max|null_expiry_count|
+------------+------------+------------+------------+-----------------+
|  2015-01-01|  2017-03-31|  2016-04-19|  2036-10-15|                0|
+------------+------------+------------+------------+-----------------+



In [0]:
df_logs_train.selectExpr(
    "min(date) as log_date_min",
    "max(date) as log_date_max"
).show()

+------------+------------+
|log_date_min|log_date_max|
+------------+------------+
|  2017-01-01|  2017-02-28|
+------------+------------+



In [0]:
df_logs_score.selectExpr(
    "min(date) as log_date_min",
    "max(date) as log_date_max"
).show()

+------------+------------+
|log_date_min|log_date_max|
+------------+------------+
|  2017-03-01|  2017-03-31|
+------------+------------+



In [0]:
df_members.selectExpr(
    "min(registration_init_time) as reg_date_min",
    "max(registration_init_time) as reg_date_max"
).show()

+------------+------------+
|reg_date_min|reg_date_max|
+------------+------------+
|  2004-03-26|  2017-04-29|
+------------+------------+



In [0]:
from pyspark.sql.functions import isnull
import pandas as pd

def null_rate_report(df, name):
    """Null count and percentage per column."""
    total = df.count()
    rows = []
    for c in df.columns:
        null_count = df.filter(isnull(col(c))).count()
        rows.append({
            "table": name,
            "column": c,
            "total_rows": total,
            "null_count": null_count,
            "null_pct": round(null_count / total * 100, 3),
        })
    return rows

def duplicate_check(df, name, key_col="msno"):
    total    = df.count()
    distinct = df.select(key_col).distinct().count()
    print(f"  [{name}] total={total:,}  distinct {key_col}={distinct:,}  duplicate rows={total - distinct:,}")

# ── Null rate report across all six tables ──
report_rows = []
report_rows += null_rate_report(df_train,      "train")
report_rows += null_rate_report(df_members,    "members")
report_rows += null_rate_report(df_txn_train,  "transactions_train")
report_rows += null_rate_report(df_txn_score,  "transactions_score")
report_rows += null_rate_report(df_logs_train, "user_logs_train")
report_rows += null_rate_report(df_logs_score, "user_logs_score")

dq_report = pd.DataFrame(report_rows)
print("NULL RATE REPORT")
print(dq_report.to_string(index=False))

# ── Duplicate / uniqueness checks ──
print("\n DUPLICATE CHECK (msno uniqueness where expected)")
duplicate_check(df_train,   "train")        # should be ~unique
duplicate_check(df_members, "members")      # should be ~unique
print(f"  [transactions_train] distinct msno = {df_txn_train.select('msno').distinct().count():,}  (many rows/user expected)")
print(f"  [transactions_score] distinct msno = {df_txn_score.select('msno').distinct().count():,}  (many rows/user expected)")
print(f"  [user_logs_train]    distinct msno = {df_logs_train.select('msno').distinct().count():,}  (many rows/user expected)")
print(f"  [user_logs_score]    distinct msno = {df_logs_score.select('msno').distinct().count():,}  (many rows/user expected)")

# ── Cohort coverage summary (from our earlier exploration, now formalized) ──
print("\n=== COHORT COVERAGE (train_v2 labeled users) ===")
total_labeled = df_train.select("msno").distinct().count()
log_overlap   = df_train.join(df_logs_train, "msno", "inner").select(df_train.msno).distinct().count()
txn_overlap   = df_train.join(df_txn_train,  "msno", "inner").select(df_train.msno).distinct().count()

print(f"  Total labeled users              : {total_labeled:,}")
print(f"  With log activity (Jan-Feb)       : {log_overlap:,}  ({log_overlap/total_labeled*100:.1f}%)")
print(f"  With transaction activity         : {txn_overlap:,}  ({txn_overlap/total_labeled*100:.1f}%)")

=== NULL RATE REPORT ===
             table                 column  total_rows  null_count  null_pct
             train                   msno      970960           0     0.000
             train               is_churn      970960           0     0.000
           members                   msno     6769473           0     0.000
           members                   city     6769473           0     0.000
           members                     bd     6769473           0     0.000
           members                 gender     6769473     4429505    65.434
           members         registered_via     6769473           0     0.000
           members registration_init_time     6769473           0     0.000
transactions_train                   msno     1771849           0     0.000
transactions_train      payment_method_id     1771849           0     0.000
transactions_train      payment_plan_days     1771849           0     0.000
transactions_train        plan_list_price     1771849          

In [0]:
from pyspark.sql import functions as F
# Investigate gender nulls — is it random or systematic?
print("Gender value counts (including nulls):")
df_members.groupBy("gender").count().orderBy("count", ascending=False).show()

print("\nDoes gender-null rate vary by registration channel?")
df_members.groupBy("registered_via").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("gender").isNull(), 1).otherwise(0)).alias("null_gender_count")
).withColumn(
    "null_pct", F.round(F.col("null_gender_count") / F.col("total") * 100, 1)
).orderBy("registered_via").show()

Gender value counts (including nulls):
+------+-------+
|gender|  count|
+------+-------+
|  NULL|4429505|
|  male|1195355|
|female|1144613|
+------+-------+


Does gender-null rate vary by registration channel?
+--------------+-------+-----------------+--------+
|registered_via|  total|null_gender_count|null_pct|
+--------------+-------+-----------------+--------+
|            -1|      1|                1|   100.0|
|             1|     43|               39|    90.7|
|             2|   1452|             1281|    88.2|
|             3|1643208|           431740|    26.3|
|             4|2793213|          2536922|    90.8|
|             5|   3115|             2978|    95.6|
|             6|   1213|             1155|    95.2|
|             7| 805895|           691041|    85.7|
|             8|   3982|             3598|    90.4|
|             9|1482863|           732150|    49.4|
|            10|     10|               10|   100.0|
|            11|  25047|            19837|    79.2|
|       

### Data Quality Finding — `members.gender` nulls are systematic, not random

65.4% of `gender` values are null. Breaking this down by `registered_via` shows the
null rate ranges from 26.3% to 100% depending on registration channel — this is
**not missing at random**. Certain registration channels appear to never collect gender, while others collect it for the
majority of users.

**Decision:** we do not impute gender. In Step 3 feature engineering, we encode it
as a 3-way categorical (`male` / `female` / `unknown`) rather than filling nulls,
since "unknown" carries real information (registration channel), not noise.
I'll also evaluate in Step 4 whether `gender` adds incremental signal beyond what
`registered_via` already captures, since the two are highly correlated.

In [0]:
def write_bronze(df, table_name):
    full_name = f"{BRONZE_DB}.{table_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    print(f"  ✓ Written: {full_name}  ({df.count():,} rows)")

print("Writing Bronze Delta tables...")
write_bronze(df_train,      "train_v2")
write_bronze(df_members,    "members")
write_bronze(df_txn_train,  "transactions_train")
write_bronze(df_txn_score,  "transactions_score")
write_bronze(df_logs_train, "user_logs_train")
write_bronze(df_logs_score, "user_logs_score")

print(f"\nStep 1 complete. Bronze tables available at:")
for t in ["train_v2", "members", "transactions_train", "transactions_score", "user_logs_train", "user_logs_score"]:
    print(f"  {BRONZE_DB}.{t}")

Writing Bronze Delta tables...
  ✓ Written: churn_project.bronze.train_v2  (970,960 rows)
  ✓ Written: churn_project.bronze.members  (6,769,473 rows)
  ✓ Written: churn_project.bronze.transactions_train  (1,771,849 rows)
  ✓ Written: churn_project.bronze.transactions_score  (1,431,009 rows)
  ✓ Written: churn_project.bronze.user_logs_train  (24,884,219 rows)
  ✓ Written: churn_project.bronze.user_logs_score  (18,396,362 rows)

Step 1 complete. Bronze tables available at:
  churn_project.bronze.train_v2
  churn_project.bronze.members
  churn_project.bronze.transactions_train
  churn_project.bronze.transactions_score
  churn_project.bronze.user_logs_train
  churn_project.bronze.user_logs_score


In [0]:
for t in ["train_v2", "members", "transactions_train", "transactions_score", "user_logs_train", "user_logs_score"]:
    print(f"\n── {BRONZE_DB}.{t} ──")
    spark.table(f"{BRONZE_DB}.{t}").printSchema()
    spark.table(f"{BRONZE_DB}.{t}").show(3, truncate=False)


── churn_project.bronze.train_v2 ──
root
 |-- msno: string (nullable = true)
 |-- is_churn: integer (nullable = true)

+--------------------------------------------+--------+
|msno                                        |is_churn|
+--------------------------------------------+--------+
|ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=|1       |
|f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=|1       |
|zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=|1       |
+--------------------------------------------+--------+
only showing top 3 rows

── churn_project.bronze.members ──
root
 |-- msno: string (nullable = true)
 |-- city: integer (nullable = true)
 |-- bd: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- registered_via: integer (nullable = true)
 |-- registration_init_time: date (nullable = true)

+--------------------------------------------+----+---+------+--------------+----------------------+
|msno                                        |city|bd |gender|registered_via

## Final Summary and Accomplishments 

- Explored all six raw files before assuming anything about their structure
- Made an evidence-based decision on the log lookback window (60 days, not 90, based on coverage curves)
- Trimmed `user_logs.csv` and `transactions.csv` locally with DuckDB (28 GB → manageable size in 18 seconds)
- Defined explicit schemas only after inspecting real data.
- Found and handled real anomalies: epoch placeholder dates, far-future expiry dates, zero-payment transaction patterns, and a systematic (not random) null pattern in gender
- Parsed all dates from `yyyyMMdd` integers to proper `DateType`
- Produced a full null-rate and duplicate-check Data Quality report
- Wrote six validated Bronze Delta tables: `train_v2`, `members`, `transactions_train`, `transactions_score`, `user_logs_train`, `user_logs_score`